# Aralin 08 - Disenyong Pattern ng Multi-Agent


## Setup


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Bakit Multi-Agent Systems?

Ang mga totoong gawain tulad ng pagpaplano ng paglalakbay ay nangangailangan ng iba't ibang uri ng kadalubhasaan — logistics, lokal na kaalaman, pagba-budget, at iba pa. Ang isang agent na sumusubok hawakan ang lahat ay mabilis na nagiging mahirap kontrolin.

Nilulutas ng multi-agent systems ito sa pamamagitan ng **specialization**: bawat agent ay nakatuon sa isang larangan ng kadalubhasaan, na nagbibigay ng mas mataas na kalidad na mga resulta kaysa sa isang generalist. Pinapabuti din nila ang **scalability** — maaari kang magdagdag ng mga bagong agent (hal., isang flight specialist, isang restaurant critic) nang hindi nire-rewrite ang umiiral na workflow. Ang mga agent ay nagtutulungan sa pamamagitan ng isang istrukturadong pipeline, na ipinapasa ang konteksto mula sa isa patungo sa susunod.


## Paglikha ng mga Espesyal na Ahente


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Paggawa ng Isang Sunud-sunod na Daloy ng Trabaho

Pinapayagan ka ng `WorkflowBuilder` na ikonekta ang mga ahente sa isang direktadong grap. Dito, gumagawa tayo ng isang simpleng dalawang-hakbang na pipeline: ang **TravelPlanner** ang gumagawa ng balangkas ng itineraryo, pagkatapos ay sinusuri at pinapaganda ito ng **TravelConcierge**.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Pagdaragdag ng Higit pang mga Ahente sa Workflow

Isa sa pinakamalaking bentahe ng multi-agent pattern ay kung gaano ito kadaling palawakin. Sa ibaba, nagdagdag tayo ng isang **BudgetReviewer** na ahente na sinusuri ang plano laban sa budget ng biyahero, tinutukoy ang mga item na maaaring magdulot ng sobra sa limitasyon ng gastos, at nagmumungkahi ng mga alternatibong makakatipid ng pera. Ang workflow ay ngayo'y nagpapatakbo ng tatlong ahente nang sunud-sunod:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

Sa leksyong ito natutunan mo kung paano:

1. **Lumikha ng mga espesyal na ahente** — bawat isa ay may nakatuong papel (pagpaplano, concierge, pagsusuri ng budget).
2. **Ikabit ang mga ahente sa isang sunod-sunod na workflow** gamit ang `WorkflowBuilder` at `add_edge`.
3. **I-stream ang output** mula sa isang multi-agent pipeline, sinusubaybayan kung aling ahente ang nagsasalita.
4. **Palawakin ang workflow** sa pamamagitan ng pagdagdag ng mga bagong ahente sa chain nang hindi inaayos ang mga umiiral na ahente.

Pinananatili ng multi-agent design pattern ang bawat ahente na simple habang nagbubunga ng mas mayaman, mas masusing nirerebyu na mga resulta kaysa kaya ng isang ahente lamang.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Paunawa**:
Ang dokumentong ito ay naisalin gamit ang AI translation service na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagamat nagsusumikap kaming maging tumpak, pakatandaan na maaaring maglaman ang mga awtomatikong salin ng mga pagkakamali o maling impormasyon. Ang orihinal na dokumento sa kanyang sariling wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot para sa anumang hindi pagkakaunawaan o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
